In [1]:
import geopandas as gpd
import pandas as pd

In [4]:
import geopandas as gpd
import pandas as pd

geom = gpd.read_file('src/Vectorisation/')
geom = geom.reset_index()
geom.rename(columns={'index': 'geom_id'}, inplace=True)
df = pd.read_excel('src/legende_renove_2.xlsx')
# only one entry with no folio, we can drop it (only information is in column "*": 'MANQUE DES NUMEROS POUR FO. 10 (IL MANQUE PROBABLEMENT UNE PAGE)')
df = df[df['folio'].notnull()]
points = gpd.read_file('src/numeros_merged-v2.json') # no null values in the relevant part. 
print(len(points))
points.drop(columns=['id'], inplace=True)
points = points.drop_duplicates()
print(len(points))

def check_decimal_vals(df, col) -> None:
    tdf = df.copy()
    tdf['decimal_part'] = tdf.apply(lambda x: x[col] - int(x[col]), axis=1)
    dec_vals = list(tdf.decimal_part.value_counts().items())
    if len(dec_vals) > 1 or (len(dec_vals) == 1 and dec_vals[0][0] != 0.0):
        print(f'There are {col} with decimal parts. This is unexpected.')
        print(dec_vals)
        print('Please check the data.')

def number_to_parcel_id(number: float) -> str:
    str_number = str(number)
    if '.' not in str_number:
        return str_number
    vals = [v.strip() for v in str(number).split('.')]
    if len(vals) == 1:
        print(f'Unexpected value: {number}.')
        return vals[0]
    main_part = vals[0]
    decim_part = vals[1]
    if decim_part == '0':
        return main_part
    return f'{main_part}-{decim_part}'

def fix_numerical_error(v:float) -> str:
    return str(float(int(10*float(v)))/(10.))

check_decimal_vals(df, 'folio')
check_decimal_vals(points, 'folio')
df['parcel_id'] = df['nr'].apply(number_to_parcel_id)
df['folio'] = df['folio'].astype(int).astype(str)
points['folio'] = points['folio'].astype(int).astype(str)
points['num'] = points['num'].apply(fix_numerical_error) # fix numerical error
points['parcel_id'] = points['num'].apply(number_to_parcel_id)
df['merge_id'] = df['folio'] + 'f' + df['parcel_id']
points['merge_id'] = points['folio'] + 'f' + points['parcel_id']


13887
13887


In [2]:
from shapely.geometry import Point, Polygon
from tqdm.notebook import tqdm
tqdm.pandas()

def locate_parcel_geom_id(geom: list[tuple[str, Polygon]], point: Point) -> str:
    for id, geometry in geom:
        if geometry.contains(point):
            return id
    return None

geom_dict = list(geom.set_index('geom_id')['geometry'].to_dict().items())
geom_dict

# this part is slow, around 25 to 30 minutes. Optimisation possible according to that link, but require numba: https://stackoverflow.com/questions/36399381/whats-the-fastest-way-of-checking-if-a-point-is-inside-a-polygon-in-python
points['geom_id'] =  points.progress_apply(lambda x: locate_parcel_geom_id(geom_dict, x.geometry), axis=1)
points['geom_id'] = points['geom_id'].apply(lambda x: str(int(x)) if not pd.isna(x) else None)

  0%|          | 0/13887 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [13]:
# so we don't need to recompute the geometry and to which polygon they belong. 
points.to_crs('EPSG:4326').to_file('src/numeros_merged-v3.geojson', driver='GeoJSON')
geom.to_crs('EPSG:4326').to_file('src/geometries_with_index.geojson', driver='GeoJSON')

In [14]:
num_df = gpd.read_file('src/numeros_merged-v3.geojson')
num_df.drop(columns=['id'], inplace=True)
num_df = num_df.drop_duplicates()
gdf = gpd.read_file('src/geometries_with_index.geojson')

In [5]:
num_df.columns, gdf.columns, df.columns

(Index(['num', 'folio', 'parcel_id', 'merge_id', 'geom_id', 'geometry'], dtype='object'),
 Index(['geom_id', 'class', 'layer', 'pg_nbr', 'folio_nbr', 'geometry'], dtype='object'),
 Index(['*', 'folio', 'nr', 'articles', 'Noms locaux', 'ares', 'cent',
        'prix proportionnels par are de la commission cadastrale', 'use',
        'owner', 'parcel_id', 'merge_id'],
       dtype='object'))

In [16]:
# Merge df (legend) with num_df (numbers with geom_id) on merge_id
merged_df = df.merge(num_df[['merge_id', 'geom_id', 'geometry']], on='merge_id', how='left')

# Ensure geom_id is consistent type (string) before merging
merged_df['geom_id'] = merged_df['geom_id'].astype(str)
gdf['geom_id'] = gdf['geom_id'].astype(str)

# Merge with gdf (geometries) on geom_id to get parcel geometries
merged_df = merged_df.merge(gdf[['geom_id', 'geometry']], on='geom_id', how='left', suffixes=('_point', '_parcel'))

# Group by the primary key (assuming "*" column is the parcel identifier) to handle multiple numbers per parcel
# Aggregate geometries and numbers if one parcel has multiple entries
df_final = merged_df.groupby('*').agg({
    'use': 'first',  # use should be same for all entries of same parcel
    'folio': 'first',
    'nr': lambda x: list(x.dropna()),  # collect all parcel numbers
    'geometry_parcel': 'first',  # parcel geometry (should be same)
    'geometry_point': lambda x: list(x.dropna()),  # collect all point geometries
}).reset_index()

# Keep only the parcel geometry as the main geometry
df_final = df_final.rename(columns={'geometry_parcel': 'geometry'})
df_final = df_final.drop(columns=['geometry_point'], errors='ignore')

print(f"Created df_final with {len(df_final)} rows")
print(f"Columns: {df_final.columns.tolist()}")
df_final.head()

Created df_final with 13515 rows
Columns: ['*', 'use', 'folio', 'nr', 'geometry']


,*,use,folio,nr,geometry
0,1,Magasin sur domaine public,1,[1.0],"POLYGON ((6.63308 46.52293, 6.63305 46.52294, ..."
1,2,Magasin sur domaine public,1,[2.0],"POLYGON ((6.63305 46.52292, 6.63308 46.52292, ..."
2,3,Magasin sur domaine public,1,[3.0],"POLYGON ((6.63305 46.52286, 6.63307 46.52286, ..."
3,4,Place,1,[4.0],"POLYGON ((6.63309 46.523, 6.63319 46.52302, 6...."
4,5,Couvert de fontaine,1,[5.0],"POLYGON ((6.63359 46.52284, 6.63365 46.52283, ..."


In [17]:
import re
import unicodedata


def normalize_use_text(value: str) -> str:
    if pd.isna(value):
        return ""
    txt = str(value).strip().lower()
    txt = "".join(
        c for c in unicodedata.normalize("NFD", txt)
        if unicodedata.category(c) != "Mn"
    )
    txt = re.sub(r"[^a-z0-9\s]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


agri_patterns = [
    ("etable", [r"\betable\b", r"\betables\b", r"\bporc\b", r"\bporcs\b"]),
    ("ecurie", [r"\becurie\b", r"\becuries\b"]),
    ("grange", [r"\bgrange\b", r"\bgranges\b", r"\bgrance\b"]),
    ("fenil", [r"\bfenil\b", r"\bfenils\b"]),
    ("grenier", [r"\bgrenier\b", r"\bgreniers\b"]),
    ("hangar", [r"\bhangar\b", r"\bhangars\b"]),
    ("champ", [r"\bchamp\b", r"\bchamps\b"]),
    ("pre", [r"\bpre\b", r"\bpres\b"]),
    ("paturage", [r"\bpaturage\b"]),
    ("vigne", [r"\bvigne\b", r"\bvignes\b"]),
    ("verger", [r"\bverger\b", r"\bvergers\b"]),
    ("cheneviere", [r"\bcheneviere\b", r"\bchenevieres\b"]),
    ("moulin", [r"\bmoulin\b", r"\bmoulins\b"]),
    ("pressoir", [r"\bpressoir\b", r"\bpressoirs\b"]),
    ("plantage", [r"\bplantage\b", r"\bplantages\b"]),
    ("place_a_fumier", [r"\bfumier\b"]),
    ("voliere", [r"\bvoliere\b", r"\bvolieres\b"]),
    ("abattoir", [r"\babattoir\b", r"\babattoirs\b", r"\btuerie\b"]),
    ("serre", [r"\bserre\b", r"\bserres\b"]),
]


def map_agri_category(use_value: str):
    normalized = normalize_use_text(use_value)
    for category, patterns in agri_patterns:
        if any(re.search(pattern, normalized) for pattern in patterns):
            return category
    return None


def split_use_string(use_val):
    if pd.isna(use_val):
        return []
    if isinstance(use_val, list):
        return use_val
    use_str = str(use_val).strip()
    if not use_str:
        return []
    return [item.strip() for item in use_str.split(',')]


# Split use into lists
df_agri = df_final.copy()
df_agri['use_list'] = df_agri['use'].apply(split_use_string)

# Explode to handle cases where use has multiple comma-separated values
agri_long = df_agri.explode('use_list').copy()
agri_long['use_raw'] = agri_long['use_list'].astype(str).str.strip()
agri_long['agri_category'] = agri_long['use_raw'].apply(map_agri_category)

# Filter to only agriculture uses
agri_long = agri_long[agri_long['agri_category'].notna()].copy()

# Group back by parcel to create lists of agriculture categories
agri_by_parcel = agri_long.groupby('*').agg(
    agri_categories=('agri_category', lambda s: sorted(set(s))),
    use_agri=('use_raw', lambda s: sorted(set(s))),
)

# Create final agriculture dataset with one row per parcel
agri_selected = df_final[df_final['*'].isin(agri_by_parcel.index)].copy()
agri_selected = agri_selected.set_index('*').join(agri_by_parcel).reset_index()

# Rename columns for clarity
agri_selected['use_all'] = agri_selected['use']
agri_selected['use'] = agri_selected['use_agri']
agri_selected = agri_selected.drop(columns=['use_agri'])

# Export to CSV
agri_selected.to_csv('lausanne-1888-cadastre-renove-registre-agriculture-merged.csv', index=False)

print(f"Agriculture dataset: {len(agri_selected)} parcels")
print(f"Merged agriculture categories: {sorted(agri_long['agri_category'].unique())}")
agri_selected[['*', 'use', 'use_all', 'agri_categories']].head(10)

Agriculture dataset: 4304 parcels
Merged agriculture categories: ['abattoir', 'champ', 'ecurie', 'etable', 'fenil', 'grange', 'grenier', 'hangar', 'moulin', 'place_a_fumier', 'pre', 'pressoir', 'serre', 'vigne', 'voliere']


,*,use,use_all,agri_categories
0,14,[Volière],Volière,[voliere]
1,19,[grenier],"Atelier, grenier",[grenier]
2,76,[Hangar],Hangar,[hangar]
3,95,[pressoir],"Maison d'habitation, cave, pressoir",[pressoir]
4,179,[pressoir],"Maison d'habitation, bureaux, pressoir",[pressoir]
5,255,[Hangar],Hangar,[hangar]
6,345,[Hangar],"Hangar, chambre à lessive",[hangar]
7,359,[Hangar],Hangar,[hangar]
8,385,[Hangar],"Hangar, atelier",[hangar]
9,407,[Hangar],Hangar,[hangar]


In [18]:
# Verify etable variants are properly merged
print("Etable variants grouped together:")
print(sorted(agri_long.loc[agri_long['agri_category'] == 'etable', 'use_raw'].unique()))

Etable variants grouped together:
['Abattoir des porcs', 'Ecurie des porcs', 'Etable à porcs', 'Etable à porcs (terrasse)', 'Etable à porcs construite par Tivel', 'Etables à porcs', 'abattoir des porcs', 'étable à porcs', 'étables à porcs']


In [19]:
agri_selected.columns

Index(['*', 'use', 'folio', 'nr', 'geometry', 'agri_categories', 'use_all'], dtype='object')

In [20]:
# Convert to GeoDataFrame and save as GeoJSON
agri_gdf = gpd.GeoDataFrame(agri_selected, geometry='geometry', crs='EPSG:4326')
agri_gdf.to_file('lausanne-1888-cadastre-renove-registre-agriculture-merged.geojson', driver='GeoJSON')
print(f"Saved {len(agri_gdf)} agriculture parcels to GeoJSON")

Saved 4304 agriculture parcels to GeoJSON
